# Geocoding — Parcel Address Enrichment
**Ontario Distribution-Connected Solar Siting | 10+ MW Projects**

> **Standalone utility notebook** — runs independently from the phase pipeline.
> Can be executed any time after Phase 3 has completed for a county.

Reverse-geocode the centroid of each developable parcel using the Mapbox
Geocoding API (v6) to obtain structured address data. Assign a unique parcel
identifier. **Updates the existing `developable_parcels_{slug}` table in-place**
by adding new columns — no separate output table is created.

**Input / Output (same table)**
- `analysis.developable_parcels_{county_slug}` (Phase 3 output)

**Columns Added**
| Column | Type | Source |
|---|---|---|
| `solar_parcel_uid` | TEXT | UUID-5 from parcel identity fields |
| `lat` / `lon` | FLOAT | WGS84 centroid via `ST_PointOnSurface` |
| `main_address` | TEXT | Mapbox primary reverse-geocoded address |
| `secondary_addresses` | TEXT (JSON) | Nearby addresses array |
| `county` / `township` / `locality` / `province` | TEXT | Mapbox context hierarchy |

---
Change only `COUNTY_NAME` in the Configuration cell to reproduce for any county.

## 0 · Configuration & Dependencies

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# PROJECT PARAMETERS — edit this cell only to run for a different county
# ─────────────────────────────────────────────────────────────────────────────

import os

COUNTY_NAME = "Ottawa"          # Must match adm_district_township.name_2

# PostGIS connection
PG_CONN = os.environ["POSTGRES_CONNECTION_STRING"]

# Output schema — tables are named per step and county automatically
OUTPUT_SCHEMA = "analysis"

# CRS
CRS_WGS84   = 4326
CRS_NAD83   = 4269   # NAD83 geographic — bridge CRS
CRS_ONTARIO = 5321   # NAD83(CSRS) / Ontario MNR Lambert — all calculations

# Geocoding parameters
GEOCODE_BATCH_DELAY = 0.05      # seconds between API calls (≤ 600 req/min on free tier)
GEOCODE_ENDPOINT    = "https://api.mapbox.com/search/geocode/v6/reverse"

## 1 · Environment Setup & Utilities

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import uuid
import time
import json

import geopandas as gpd
import pandas as pd
import requests
from sqlalchemy import create_engine, text
from dotenv import load_dotenv

engine = create_engine(PG_CONN)

# Load Mapbox token from environment
load_dotenv()
MAPBOX_TOKEN = os.environ["MAPBOX_TOKEN"]


def read_postgis(sql: str, geom_col: str = "geom") -> gpd.GeoDataFrame:
    """Execute a PostGIS query and return a WGS84 GeoDataFrame."""
    gdf = gpd.read_postgis(sql, engine, geom_col=geom_col)
    if gdf.crs is None:
        gdf = gdf.set_crs(epsg=CRS_WGS84)
    return gdf.to_crs(epsg=CRS_WGS84)


def save_to_postgis(gdf: gpd.GeoDataFrame, table: str, label: str) -> None:
    """Write a GeoDataFrame to PostGIS in EPSG:5321 and create a GiST spatial index."""
    geom_col = gdf.geometry.name
    gdf.to_crs(epsg=CRS_ONTARIO).to_postgis(
        name=table,
        con=engine,
        schema=OUTPUT_SCHEMA,
        if_exists="replace",
        index=False,
        chunksize=500
    )
    with engine.connect() as conn:
        conn.execute(text(
            f"CREATE INDEX IF NOT EXISTS {table}_geom_idx "
            f"ON {OUTPUT_SCHEMA}.{table} USING GIST({geom_col})"
        ))
        conn.commit()
    print(f"  {label:35s} → {OUTPUT_SCHEMA}.{table} "
          f"({len(gdf):,} rows, EPSG:{CRS_ONTARIO}, GiST index)")


county_slug = COUNTY_NAME.lower().replace(" ", "_")

with engine.connect() as conn:
    print("PostGIS:", conn.execute(text("SELECT PostGIS_Full_Version()")).scalar())
print(f"County : {COUNTY_NAME} (slug: {county_slug})")
print(f"Mapbox token loaded: {'Yes' if MAPBOX_TOKEN else 'No'}")

PostGIS: POSTGIS="3.3.4 0" [EXTENSION] PGSQL="150" GEOS="3.11.2-CAPI-1.17.2" SFCGAL="SFCGAL 1.4.1, CGAL 5.5.2, BOOST 1.82.0" PROJ="9.2.1" GDAL="GDAL 3.6.4, released 2023/04/17" LIBXML="2.11.4" LIBJSON="0.16" LIBPROTOBUF="1.4.1" WAGYU="0.5.0 (Internal)" TOPOLOGY RASTER
County : Ottawa (slug: ottawa)
Mapbox token loaded: Yes


---
## Step 1 — Load Developable Parcels & Compute Centroids

Read the Phase 3 output and compute a WGS84 centroid per parcel. Uses
`ST_PointOnSurface` (guaranteed inside the polygon) rather than `ST_Centroid`
(which can fall outside concave shapes).

In [14]:
county_slug = COUNTY_NAME.lower().replace(" ", "_")

# Join developable_parcels (clean parcel outlines) with developable_area_parcels
# (lot-level attributes) to enrich the table for UID generation and geocoding
gdf_parcels = read_postgis(f"""
SELECT
    dp.parcel_id,
    dp.parcel_acre,
    dap.lot_ident,
    dap.concession_ident,
    dap.geographic_township_name,
    dap.land_use_designation,
    dap.station_name,
    dap.ldc_name,
    ST_Y(ST_Transform(ST_PointOnSurface(dp.geom), {CRS_WGS84})) AS lat,
    ST_X(ST_Transform(ST_PointOnSurface(dp.geom), {CRS_WGS84})) AS lon,
    ST_Transform(dp.geom, {CRS_WGS84}) AS geom
FROM {OUTPUT_SCHEMA}.developable_parcels_{county_slug} dp
LEFT JOIN (
    SELECT DISTINCT ON (parcel_id)
        parcel_id, lot_ident, concession_ident,
        geographic_township_name, land_use_designation,
        station_name, ldc_name
    FROM {OUTPUT_SCHEMA}.developable_area_parcels_{county_slug}
    ORDER BY parcel_id
) dap ON dp.parcel_id = dap.parcel_id
""")

print(f"Step 1 — Loaded developable parcels for '{COUNTY_NAME}':")
print(f"  Row count: {len(gdf_parcels):,}")
print(f"  Columns  : {list(gdf_parcels.columns)}")
gdf_parcels.head(10)

Step 1 — Loaded developable parcels for 'Ottawa':
  Row count: 57
  Columns  : ['parcel_id', 'parcel_acre', 'lot_ident', 'concession_ident', 'geographic_township_name', 'land_use_designation', 'station_name', 'ldc_name', 'lat', 'lon', 'geom']


,parcel_id,parcel_acre,lot_ident,concession_ident,geographic_township_name,land_use_designation,station_name,ldc_name,lat,lon,geom
0,26348,276.313960,LOT 12,CON 11,GOULBOURN,Rural Countryside,Janet King DS 28kV,Hydro Ottawa Limited,45.222118,-75.994162,"MULTIPOLYGON (((-75.98466 45.21846, -75.99688 ..."
1,26451,407.104933,LOT 9,CON 9,GOULBOURN,Rural Countryside,Janet King DS 28kV,Hydro Ottawa Limited,45.192342,-75.991633,"MULTIPOLYGON (((-75.98752 45.2, -75.98827 45.2..."
2,26521,178.910278,LOT 10,CON 10,GOULBOURN,Rural Countryside,Janet King DS 28kV,Hydro Ottawa Limited,45.204682,-75.993712,"MULTIPOLYGON (((-75.99589 45.21061, -76.00156 ..."
3,26614,114.077670,LOT 5,CON 8,GOULBOURN,Rural Countryside,Janet King DS 28kV,Hydro Ottawa Limited,45.168483,-75.997909,"MULTIPOLYGON (((-76.00408 45.1743, -76.00599 4..."
4,26841,98.890162,LOT 4,CON 9,GOULBOURN,Rural Countryside,Janet King DS 28kV,Hydro Ottawa Limited,45.175829,-76.017271,"MULTIPOLYGON (((-76.01095 45.17033, -76.0104 4..."
5,26876,198.776409,LOT 5,CON 9,GOULBOURN,Rural Countryside,Janet King DS 28kV,Hydro Ottawa Limited,45.178779,-76.013226,"MULTIPOLYGON (((-76.00786 45.17226, -76.0043 4..."
6,26891,161.141970,LOT 4,CON 9,GOULBOURN,Rural Countryside,Janet King DS 28kV,Hydro Ottawa Limited,45.172670,-76.021505,"MULTIPOLYGON (((-76.01707 45.16935, -76.01879 ..."
7,29792,95.325264,LOT 21,CON 2,MARCH,Rural Countryside,Marchwood MS,Hydro Ottawa Limited,45.382876,-76.011929,"MULTIPOLYGON (((-76.01145 45.38669, -76.01168 ..."
8,29825,86.368392,LOT 23,CON 3,MARCH,Rural Countryside,Marchwood MS,Hydro Ottawa Limited,45.399882,-76.005354,"MULTIPOLYGON (((-76.00193 45.40397, -76.01229 ..."
9,29840,186.268739,LOT 25,CON 2,MARCH,Rural Countryside,Marchwood MS,Hydro Ottawa Limited,45.400356,-76.030786,"MULTIPOLYGON (((-76.02721 45.4062, -76.03077 4..."


---
## Step 2 — Generate Deterministic Parcel UIDs

Create a deterministic **unique parcel identifier** (`solar_parcel_uid`) by
hashing the county slug, lot identity, concession, and parcel ID using UUID-5.
This is distinct from the cadastre `parcel_id` / `parcel_ogf_id` and is stable
across re-runs — the same input always produces the same UUID.

In [15]:
def make_parcel_uid(row):
    """Deterministic UUID-5 from parcel identity fields."""
    key = f"{county_slug}|{row['lot_ident']}|{row['concession_ident']}|{row['parcel_id']}"
    return str(uuid.uuid5(uuid.NAMESPACE_URL, key))

gdf_parcels["solar_parcel_uid"] = gdf_parcels.apply(make_parcel_uid, axis=1)

print(f"Step 2 — Generated {len(gdf_parcels):,} parcel UIDs")
print(f"  Unique UIDs: {gdf_parcels['solar_parcel_uid'].nunique():,}")
print(f"\nSample UIDs:")
gdf_parcels[["lot_ident", "concession_ident", "parcel_id", "solar_parcel_uid"]].head(5)

Step 2 — Generated 57 parcel UIDs
  Unique UIDs: 57

Sample UIDs:


,lot_ident,concession_ident,parcel_id,solar_parcel_uid
0,LOT 12,CON 11,26348,951419ab-e07f-54ed-905a-b85bb00aa5e2
1,LOT 9,CON 9,26451,62c2b3b7-a7a5-5c31-92e7-b15e723b3cde
2,LOT 10,CON 10,26521,ae0f5e73-5719-58b1-a620-8f01e2b75990
3,LOT 5,CON 8,26614,bbf67272-00a4-5a94-a864-511f2a3a6866
4,LOT 4,CON 9,26841,022613bd-6fc2-528b-bfea-092ea68593bf


---
## Step 3 — Reverse-Geocode via Mapbox API

Call the Mapbox v6 reverse endpoint for each parcel centroid. Parse the
structured response to extract address components.

- **Primary address**: `full_address` from the first feature
- **Secondary addresses**: `full_address` from remaining features (JSON array)
- **Context fields**: `district.name` → county, `place.name` → township,
  `locality.name` → locality, `region.name` → province

Individual failures are caught and logged — the loop does not abort.
Progress is printed every 50 rows.

In [16]:
def reverse_geocode(lat, lon, token, types="address"):
    """Call Mapbox v6 reverse geocode. Returns parsed dict."""
    resp = requests.get(
        GEOCODE_ENDPOINT,
        params={
            "longitude": lon,
            "latitude": lat,
            "access_token": token,
            "types": types,
            "limit": 5,
        },
        timeout=10,
    )
    resp.raise_for_status()
    return resp.json()


def parse_features(data):
    """Extract address components from Mapbox v6 response features."""
    features = data.get("features", [])
    if not features:
        return None, None, {}
    primary   = features[0]["properties"]
    context   = primary.get("context", {})
    main_addr = primary.get("full_address", None)
    secondary = [
        f["properties"].get("full_address")
        for f in features[1:]
        if f["properties"].get("full_address")
    ]
    return main_addr, secondary, context


# Mapbox fallback strategy: try "address" first, then broaden to coarser types
# Rural parcels (farmland, forests) often have no civic address — fall back to
# place/locality/neighborhood which will still give township + county context
TYPES_PRIMARY  = "address"
TYPES_FALLBACK = "place,locality,neighborhood,address"

records = []
total = len(gdf_parcels)
fallback_count = 0

for idx, row in gdf_parcels.iterrows():
    try:
        # First attempt: address only
        data = reverse_geocode(row["lat"], row["lon"], MAPBOX_TOKEN, TYPES_PRIMARY)
        main_addr, secondary, context = parse_features(data)

        # Fallback: if no address found, widen types to include place/locality
        if main_addr is None:
            data = reverse_geocode(row["lat"], row["lon"], MAPBOX_TOKEN, TYPES_FALLBACK)
            main_addr, secondary, context = parse_features(data)
            if main_addr is not None:
                fallback_count += 1
            time.sleep(GEOCODE_BATCH_DELAY)  # extra delay for the second call

        records.append({
            "_idx":                idx,
            "main_address":        main_addr,
            "secondary_addresses": json.dumps(secondary) if secondary else None,
            "county":              context.get("district", {}).get("name"),
            "township":            context.get("place", {}).get("name"),
            "locality":            context.get("locality", {}).get("name"),
            "province":            context.get("region", {}).get("name"),
        })
    except Exception as e:
        print(f"  [{idx+1}/{total}] FAILED: {e}")
        records.append({
            "_idx":                idx,
            "main_address":        None,
            "secondary_addresses": None,
            "county":              None,
            "township":            None,
            "locality":            None,
            "province":            None,
        })

    if (idx + 1) % 50 == 0:
        print(f"  [{idx+1}/{total}] geocoded")

    time.sleep(GEOCODE_BATCH_DELAY)

print(f"\nGeocoding complete: {len(records)} parcels processed")
print(f"  Direct address hits : {sum(1 for r in records if r['main_address'] and r['_idx'] not in [])}")
print(f"  Fallback resolved   : {fallback_count}")
print(f"  Still missing       : {sum(1 for r in records if r['main_address'] is None)}")

  [50/57] geocoded

Geocoding complete: 57 parcels processed
  Direct address hits : 57
  Fallback resolved   : 0
  Still missing       : 0


---
## Step 4 — Merge Geocoding Results into GeoDataFrame

Assign geocoding columns back into `gdf_parcels` from the records list.

In [20]:
df_geo = pd.DataFrame(records).set_index("_idx")
for col in ["main_address", "secondary_addresses", "county", "township", "locality", "province"]:
    gdf_parcels[col] = df_geo[col]

geocoded_count = gdf_parcels["main_address"].notna().sum()
missing_count  = gdf_parcels["main_address"].isna().sum()

print(f"Step 4 — Merged geocoding results:")
print(f"  With main address: {geocoded_count:,}")
print(f"  Missing address  : {missing_count:,}")

gdf_parcels[["lot_ident", "parcel_id", "lat", "lon", "main_address", "county", "township"]].head(10)

Step 4 — Merged geocoding results:
  With main address: 57
  Missing address  : 0


,lot_ident,parcel_id,lat,lon,main_address,county,township
0,LOT 12,26348,45.222118,-75.994162,"7600 Golf Club Way, Stittsville, Ontario K0A 1...",None,Stittsville
1,LOT 9,26451,45.192342,-75.991633,"7913 Flewellyn Road, Ashton, Ontario K0A 1B0, ...",None,Ashton
2,LOT 10,26521,45.204682,-75.993712,"7945 Fernbank Road, Ashton, Ontario K0A 1B0, C...",None,Ashton
3,LOT 5,26614,45.168483,-75.997909,"2150 Dwyer Hill Road, Ashton, Ontario K0A 1B0,...",None,Ashton
4,LOT 4,26841,45.175829,-76.017271,"8587 Flewellyn Road, Ashton, Ontario K0A 1B0, ...",None,Ashton
5,LOT 5,26876,45.178779,-76.013226,"8401 Flewellyn Road, Ashton, Ontario K0A 1B0, ...",None,Ashton
6,LOT 4,26891,45.172670,-76.021505,"8647 Flewellyn Road, Ashton, Ontario K0A 1B0, ...",None,Ashton
7,LOT 21,29792,45.382876,-76.011929,"1495 Murphy Side Road, Kanata, Ontario K2W 0H3...",None,Kanata
8,LOT 23,29825,45.399882,-76.005354,"2365 Dunrobin Road, Dunrobin, Ontario K0A 1T0,...",None,Dunrobin
9,LOT 25,29840,45.400356,-76.030786,"2575 Old Second Line Road, Dunrobin, Ontario K...",None,Dunrobin


---
## Step 5 — Save Updated Table to PostGIS

Overwrite `developable_parcels_{slug}` with all original columns plus the new
geocoding columns. `save_to_postgis()` replaces the table atomically.

All original Phase 3 columns are preserved; the new columns
(`solar_parcel_uid`, `lat`, `lon`, `main_address`, `secondary_addresses`,
`county`, `township`, `locality`, `province`) are appended.

In [21]:
save_to_postgis(gdf_parcels, f"developable_parcels_{county_slug}",
                "Geocoding — Parcels + addresses")

  Geocoding — Parcels + addresses     → analysis.developable_parcels_ottawa (57 rows, EPSG:5321, GiST index)


---
## Step 6 — Verification & Summary

Confirm geocoding results and verify all expected columns exist in the DB table.

In [22]:
county_slug = COUNTY_NAME.lower().replace(" ", "_")
print(f"Geocoded parcels: {len(gdf_parcels)}")
print(f"With main address:    {gdf_parcels['main_address'].notna().sum()}")
print(f"Missing address:      {gdf_parcels['main_address'].isna().sum()}")
print(f"Unique counties:      {gdf_parcels['county'].dropna().unique().tolist()}")
print(f"Unique townships:     {gdf_parcels['township'].dropna().unique().tolist()}")

# Verify columns exist in DB
with engine.connect() as conn:
    cols = conn.execute(text(f"""
        SELECT column_name
        FROM information_schema.columns
        WHERE table_schema = '{OUTPUT_SCHEMA}'
          AND table_name   = 'developable_parcels_{county_slug}'
        ORDER BY ordinal_position
    """)).fetchall()
    print(f"\nColumns in DB table ({OUTPUT_SCHEMA}.developable_parcels_{county_slug}):")
    for c in cols:
        print(f"  {c[0]}")

# Show parcels with missing addresses for retry identification
missing = gdf_parcels[gdf_parcels["main_address"].isna()]
if len(missing) > 0:
    print(f"\nParcels with missing addresses ({len(missing)}):")
    print(missing[["lot_ident", "concession_ident", "parcel_id", "lat", "lon"]].to_string())
else:
    print("\nAll parcels geocoded successfully — no missing addresses.")

Geocoded parcels: 57
With main address:    57
Missing address:      0
Unique counties:      []
Unique townships:     ['Stittsville', 'Ashton', 'Kanata', 'Dunrobin', 'Richmond', 'Smiths Falls', 'Nepean', 'Gloucester', 'Ottawa', 'Osgoode', 'Greely', 'Metcalfe', 'Vars', 'Cumberland', 'Sarsfield', 'Navan']

Columns in DB table (analysis.developable_parcels_ottawa):
  parcel_id
  parcel_acre
  lot_ident
  concession_ident
  geographic_township_name
  land_use_designation
  station_name
  ldc_name
  lat
  lon
  geom
  solar_parcel_uid
  main_address
  secondary_addresses
  county
  township
  locality
  province

All parcels geocoded successfully — no missing addresses.
